In [1]:
using LinearAlgebra
using JLD2
using Plots

In [2]:

args=[0.36,-0.6,-0.6,0.0,0.0,20.8,107.7,20.8,-107.7,6.0,70.0,3.89,-23.8,1.0,3.0,3.0]
parameters=args[1:13]
holenum=Int(args[14])
trytime=Int(args[15])
Nq=Int(args[16]);


    mt=parameters[1]
    mm=parameters[2]
    mb=parameters[3]
    Vt=parameters[4]
    ϕt=parameters[5]/180*π
    Vm=parameters[6]
    ϕm=parameters[7]/180*π
    Vb=parameters[8]
    ϕb=parameters[9]/180*π
    ϵr=parameters[10]
    Eg=parameters[11]
    θ=parameters[12]/180*π
    w=parameters[13]
  
  
  
  
  
  
  
    alattice=0.352# In units of NM
    blattice=4*π/(√3*alattice)
  
    
    
    bm=blattice*θ
    am=4*π/(√3*bm);
    constt=38.09981949*1/(mt)
    constm=38.09981949*1/(mm)
    constb=38.09981949*1/(mb) # the 38 is \hbar^2/(2me*nm^2) in units of meV
    constq=1/(Nq^2*3^(1/2)/2*am^2)*1/ϵr*9047.5636
  
    
    b1=bm*[1,0]
    b2=bm*[-1/2,√3/2]
    
  
    a1m=am*[√3/2,1/2]
    a2m=am*[0,1]
  
    
    T1=b1/(Nq)
    T2=b2/(Nq)
    
    κp=bm*[-1/2,1/(2√3)]
    κm=bm*[-1/2,-1/(2√3)]
    
    
    b1T=Int.(round.(inv([T1 T2])*b1))
    b2T=Int.(round.(inv([T1 T2])*b2))
    
    
    
    allowedq=Vector{Int64}[]
    for ja in 0:Nq-1,jb in 0:Nq-1
        push!(allowedq,[ja,jb])
    end
  
    
    
    
    wave=Vector{Int64}[]
    cutoff=18
    cutoffstandard=3.01*bm
    for ja in -cutoff:cutoff, jb in -cutoff:cutoff
        gtest=ja*b1+jb*b2;
        if (gtest[1]^2+gtest[2]^2)<cutoffstandard^2
            push!(wave,ja*b1T+jb*b2T)
        end
    end
    dimension=3*length(wave)
   
    shift=1/2*T1+1/2*T2
  
  
  
  
    single_Ham=[[zeros(ComplexF64,dimension,dimension) for _ in 1:2] for _ in 1:Nq^2]
    single_MoirePo=zeros(ComplexF64,dimension,dimension)
    tunnel=zeros(ComplexF64,dimension,dimension)
    M_tunnel=zeros(ComplexF64,dimension,dimension)
    seed_tunnel=zeros(ComplexF64,dimension,dimension)
    
    single_eigenvalue=[[zeros(Float64,dimension) for _ in 1:2] for _ in 1:Nq^2]
    single_eigenvector=[[zeros(ComplexF64,dimension,dimension) for _ in 1:2] for _ in 1:Nq^2]
    uncoupled_eigenvector=[[zeros(ComplexF64,dimension,dimension) for _ in 1:2] for _ in 1:Nq^2]
    seed_eigenvector=[[zeros(ComplexF64,dimension,dimension) for _ in 1:2] for _ in 1:Nq^2]
   
    
    for jc in eachindex(wave)
      pos=findfirst(item->item==wave[jc]-b1T,wave)
      if pos≠nothing
        single_MoirePo[3*(jc-1)+1:3*jc,3*(pos-1)+1:3*pos]=diagm([Vt*exp(im*ϕt),Vm*exp(im*ϕm),Vb*exp(im*ϕb)])
      end
      
    
      pos=findfirst(item->item==wave[jc]+b2T+b1T,wave)
      if pos≠nothing
        single_MoirePo[3*(jc-1)+1:3*jc,3*(pos-1)+1:3*pos]=diagm([Vt*exp(im*ϕt),Vm*exp(im*ϕm),Vb*exp(im*ϕb)])
      end
    
      pos=findfirst(item->item==wave[jc]-b2T,wave)
      if pos≠nothing
        single_MoirePo[3*(jc-1)+1:3*jc,3*(pos-1)+1:3*pos]=diagm([Vt*exp(im*ϕt),Vm*exp(im*ϕm),Vb*exp(im*ϕb)])
      end
    end
    
    
    single_MoirePo=single_MoirePo+single_MoirePo'
    
    for jc in eachindex(wave)
     
      tunnel[3*(jc-1)+3,3*(jc-1)+2]=w
    
      pos=findfirst(item->item==wave[jc]+b2T+b1T,wave)
      if pos≠nothing
     
        tunnel[3*(jc-1)+3,3*(pos-1)+2]=w
      end
    
      pos=findfirst(item->item==wave[jc]+b2T,wave)
      if pos≠nothing
       
        tunnel[3*(jc-1)+3,3*(pos-1)+2]=w
      end
    end
    
    tunnel=tunnel+tunnel'
    
    for jc in eachindex(wave)
  
      M_tunnel[3*(jc-1)+3,3*(jc-1)+2]=w
    
      pos=findfirst(item->item==wave[jc]-b2T-b1T,wave)
      if pos≠nothing
       
        M_tunnel[3*(jc-1)+3,3*(pos-1)+2]=w
      end
    
      pos=findfirst(item->item==wave[jc]-b2T,wave)
      if pos≠nothing
   
        M_tunnel[3*(jc-1)+3,3*(pos-1)+2]=w
      end
    end
    
    M_tunnel=M_tunnel+M_tunnel'
  
    for jc in eachindex(wave)
     
      seed_tunnel[3*(jc-1)+1,3*(jc-1)+2]+=2.0
      seed_tunnel[3*(jc-1)+1,3*(jc-1)+3]+=2.0
    end
  
    for jc in eachindex(wave)
      pos=findfirst(item->item==wave[jc]-b1T,wave)
      if pos≠nothing
        seed_tunnel[3*(jc-1)+1:3*jc,3*(pos-1)+1:3*pos]+=diagm([1*exp(im*ϕt),1*exp(im*ϕm),1*exp(im*ϕb)])
      end
      
    
      pos=findfirst(item->item==wave[jc]+b2T+b1T,wave)
      if pos≠nothing
        seed_tunnel[3*(jc-1)+1:3*jc,3*(pos-1)+1:3*pos]+=diagm([1*exp(im*ϕt),1*exp(im*ϕm),1*exp(im*ϕb)])
      end
    
      pos=findfirst(item->item==wave[jc]-b2T,wave)
      if pos≠nothing
        seed_tunnel[3*(jc-1)+1:3*jc,3*(pos-1)+1:3*pos]+=diagm([1*exp(im*ϕt),1*exp(im*ϕm),1*exp(im*ϕb)])
      end
    end
    
    seed_tunnel=seed_tunnel+seed_tunnel'
    
    
    
    
    Threads.@threads for ja in 1:Nq^2
      
      
      k=allowedq[ja][1]*T1+allowedq[ja][2]*T2
    
      for jb in eachindex(wave)
       single_Ham[ja][1][3*(jb-1)+1,3*(jb-1)+1]=norm(k+wave[jb][1]*T1+wave[jb][2]*T2-κp+shift)^2*constt+Eg #This is the configuration
       single_Ham[ja][1][3*(jb-1)+2,3*(jb-1)+2]=norm(k+wave[jb][1]*T1+wave[jb][2]*T2-κp+shift)^2*constm
       single_Ham[ja][1][3*(jb-1)+3,3*(jb-1)+3]=norm(k+wave[jb][1]*T1+wave[jb][2]*T2-κm+shift)^2*constb
      end
    
      for jb in eachindex(wave)
        single_Ham[ja][2][3*(jb-1)+1,3*(jb-1)+1]=norm(k+wave[jb][1]*T1+wave[jb][2]*T2+κp+shift)^2*constt+Eg
        single_Ham[ja][2][3*(jb-1)+2,3*(jb-1)+2]=norm(k+wave[jb][1]*T1+wave[jb][2]*T2+κp+shift)^2*constm
        single_Ham[ja][2][3*(jb-1)+3,3*(jb-1)+3]=norm(k+wave[jb][1]*T1+wave[jb][2]*T2+κm+shift)^2*constb
       end
      
    
       
      FFF=eigen(single_Ham[ja][1])
       uncoupled_eigenvector[ja][1]=FFF.vectors
  
      FFF=eigen(single_Ham[ja][2])
       uncoupled_eigenvector[ja][2]=FFF.vectors
      
       FFF=eigen(single_Ham[ja][1]+seed_tunnel)
       seed_eigenvector[ja][1]=FFF.vectors
  
      FFF=eigen(single_Ham[ja][2]+seed_tunnel)
       seed_eigenvector[ja][2]=FFF.vectors
      
    
    
      single_Ham[ja][1]+=single_MoirePo+tunnel
      single_Ham[ja][2]+=single_MoirePo+M_tunnel
     
      FFF=eigen(single_Ham[ja][1])
      single_eigenvalue[ja][1]=real(FFF.values)
      single_eigenvector[ja][1]=FFF.vectors
    
      FFF=eigen(single_Ham[ja][2])
      single_eigenvalue[ja][2]=real(FFF.values)
      single_eigenvector[ja][2]=FFF.vectors
  
    end
    
  
  
  
  
     
    BG_DensityMatrix=[[zeros(ComplexF64,dimension,dimension) for _ in 1:2] for _ in 1:Nq^2]
    for ja in 1:Nq^2, vi in 1:2, jb in 1:length(wave)*2
      BG_DensityMatrix[ja][vi]+=(uncoupled_eigenvector[ja][vi][:,jb]*(uncoupled_eigenvector[ja][vi][:,jb])') 
      
    end
     input_DensityMatrix=[[zeros(ComplexF64,dimension,dimension) for _ in 1:2] for _ in 1:Nq^2]
    
    for ja in 1:Nq^2, vi in 1:1
      A=10^(-1)*randn(ComplexF64,dimension,dimension)
        input_DensityMatrix[ja][vi]=A+A'
    end
  
    for ja in 1:Nq^2, vi in 2:2
      A=10^(-1)*randn(ComplexF64,dimension,dimension)
        input_DensityMatrix[ja][vi]=A+A'
    end
    
  
  
  
  
    for ja in 1:Nq^2, vi in 1:1, jb in 1:length(wave)*2
     input_DensityMatrix[ja][vi]+=(seed_eigenvector[ja][vi][:,jb]*(seed_eigenvector[ja][vi][:,jb])') 
    end
    for ja in 1:Nq^2, vi in 2:2, jb in 1:length(wave)*2-2
      input_DensityMatrix[ja][vi]+=(seed_eigenvector[ja][vi][:,jb]*(seed_eigenvector[ja][vi][:,jb])') 
    end
    for ja in 1:Nq^2, vi in 2:2, jb in length(wave)*2+1:length(wave)*2+1
      input_DensityMatrix[ja][vi]+=(seed_eigenvector[ja][vi][:,jb]*(seed_eigenvector[ja][vi][:,jb])') 
    end
  
    input_DensityMatrix-=BG_DensityMatrix
   
   


9-element Vector{Vector{Matrix{ComplexF64}}}:
 [[-0.001089195676738518 + 0.0im 0.035855566045698374 + 0.0809289140346851im … -0.03550275108830257 - 0.17468676854927076im 0.032616215013583916 - 0.05139433049072838im; 0.035855566045698374 - 0.0809289140346851im 0.14479106612875947 + 0.0im … 0.02853015687991066 + 0.11339067187873936im 0.004942232870544863 + 0.023617092874071807im; … ; -0.03550275108830257 + 0.17468676854927076im 0.02853015687991066 - 0.11339067187873936im … 0.06554442978522967 + 0.0im -0.01074189694252188 + 0.061543566855693704im; 0.032616215013583916 + 0.05139433049072838im 0.004942232870544863 - 0.023617092874071807im … -0.01074189694252188 - 0.061543566855693704im 0.12627274244554876 + 0.0im], [0.04238384676896926 + 0.0im 0.09606449620552998 + 0.040926434836915715im … -0.07666529580908599 + 0.08266970375987832im -0.02604671929067218 - 0.14471492958432364im; 0.09606449620552998 - 0.040926434836915715im -0.056818076813363594 + 0.0im … 0.11704287175612332 - 0.085085160759

In [3]:
pp=load("data_output/0.36mt-0.6mm-0.6mb0.0Vt0.0phit20.8Vm107.7phim20.8Vb-107.7phib6.0er70.0Eg3.89theta-23.8w1.0holenum11.0trytime3.0Nq.jld2")
   

Dict{String, Any} with 10 entries:
  "HFdensity"                => [1.08633e-8+2.70902e-23im 1.19176e-8-7.70312e-2…
  "parameters"               => [0.36, -0.6, -0.6, 0.0, 0.0, 20.8, 107.7, 20.8,…
  "single_eigenvalue"        => [[[-1686.8, -1640.81, -1588.06, -1480.25, -1430…
  "single_chern"             => ComplexF64[-1.00001+1.76697e-17im, 1.0+3.09221e…
  "energy"                   => -141.356
  "HF_eigenvalue"            => [[[-1750.48, -1704.36, -1651.79, -1543.76, -149…
  "bound"                    => -15.5684
  "HF_eigenvector"           => Vector{Matrix{ComplexF64}}[[[-1.12196e-13+6.539…
  "DIIS_input_DensityMatrix" => Vector{Vector{Matrix{ComplexF64}}}[[[[3.0359e-1…
  "HF_chern"                 => ComplexF64[-1.0-4.83157e-18im, 1.0+2.1397e-17im…

In [4]:
pp["HF_eigenvector"]

9-element Vector{Vector{Matrix{ComplexF64}}}:
 [[-1.121955938244938e-13 + 6.539529179903104e-13im -6.17559109690393e-14 - 1.440700797738356e-12im … -2.4653307163911903e-6 - 1.8491341503846838e-5im 2.3074228407769452e-9 + 6.624494173066943e-9im; 3.58292636768292e-11 - 2.0030763203478956e-11im -1.1243544681563519e-8 + 4.175293971912835e-9im … 4.923841576514683e-13 + 3.574682406964562e-13im -1.0007557350025754e-12 - 2.9496956931022746e-14im; … ; 0.061359550760169676 - 0.0024653218997544315im 1.131287509551545e-5 - 0.00029344985389350705im … -1.1491416247247769e-12 + 5.150766251825401e-13im -2.4461102995055226e-13 + 1.5592048906124258e-12im; 0.995064845648289 + 0.0im -0.0020604240359425336 - 0.0im … 2.356863738150916e-12 - 0.0im -1.468042124627589e-11 - 0.0im], [-5.073862541437858e-8 - 8.056388594256336e-8im 3.3905432527473456e-10 - 2.1110749430605894e-10im … 2.3128191459194135e-8 + 3.1608229859357185e-9im -2.022770444825192e-5 - 1.5187421627098124e-5im; 0.044643889322851904 + 0.0420893165

In [5]:
trial_densitymatrix=[[zeros(ComplexF64,dimension,dimension) for _ in 1:2] for _ in 1:Nq^2]


for ja in 1:Nq^2, vi in 1:2, jb in length(wave)*2:length(wave)*2
    trial_densitymatrix[ja][vi]+=(pp["HF_eigenvector"][ja][vi][:,jb]*(pp["HF_eigenvector"][ja][vi][:,jb])') 
end

In [6]:
function Densitymap(a1m::Vector{Float64},a2m::Vector{Float64},wave::Vector{Vector{Int}},input_DensityMatrix::Vector{Vector{Matrix{ComplexF64}}})::Tuple{Array{ComplexF64},Array{ComplexF64},Array{ComplexF64}}
    dimension=3*length(wave)  
       N3=50
      xgrid=zeros(Float64,N3,N3)
      ygrid=zeros(Float64,N3,N3)
      zgrid=zeros(ComplexF64,N3,N3,2,3)
      Total_Density=[zeros(ComplexF64,dimension,dimension) for _ in 1:2]
      for jk1 in 1:Nq^2
         Total_Density+=input_DensityMatrix[jk1]
      end
      
      for ja in 1:N3, jb in 1:N3
        xgrid[ja,jb]=(ja/N3*a1m[1]+jb/N3*a2m[1])
        ygrid[ja,jb]=(ja/N3*a1m[2]+jb/N3*a2m[2])
        rvec=ja/N3*a1m+jb/N3*a2m
        for vi in 1:2, Li in 1:3, jc in eachindex(wave), jd in eachindex(wave)
          gvec=[T1 T2]*(wave[jc]-wave[jd])
         zgrid[ja,jb,vi,Li]+=Total_Density[vi][3*(jc-1)+Li,3*(jd-1)+Li]*exp(im*(gvec[1]*rvec[1]+gvec[2]*rvec[2]))
        end
      
      end
  
      return xgrid,ygrid,zgrid
  end

Densitymap (generic function with 1 method)

In [7]:
xgrid,ygird,HFdensity=Densitymap(a1m,a2m,wave,trial_densitymatrix)

(ComplexF64[0.08979999695096334 + 0.0im 0.08979999695096334 + 0.0im … 0.08979999695096334 + 0.0im 0.08979999695096334 + 0.0im; 0.17959999390192669 + 0.0im 0.17959999390192669 + 0.0im … 0.17959999390192669 + 0.0im 0.17959999390192669 + 0.0im; … ; 4.400199850597203 + 0.0im 4.400199850597203 + 0.0im … 4.400199850597203 + 0.0im 4.400199850597203 + 0.0im; 4.489999847548167 + 0.0im 4.489999847548167 + 0.0im … 4.489999847548167 + 0.0im 4.489999847548167 + 0.0im], ComplexF64[0.15553815723859876 + 0.0im 0.2592302620643313 + 0.0im … 5.132759188873759 + 0.0im 5.236451293699492 + 0.0im; 0.20738420965146503 + 0.0im 0.3110763144771975 + 0.0im … 5.184605241286626 + 0.0im 5.288297346112358 + 0.0im; … ; 2.644148673056179 + 0.0im 2.7478407778819114 + 0.0im … 7.62136970469134 + 0.0im 7.725061809517072 + 0.0im; 2.6959947254690455 + 0.0im 2.7996868302947777 + 0.0im … 7.673215757104206 + 0.0im 7.7769078619299385 + 0.0im], [1.7986742597112798e-10 + 2.4135225010105117e-26im 1.9085192023287196e-10 + 7.36641028

In [14]:
(sum(HFdensity[:,:,2,3]))/50^2*1/9

0.500095015727813 + 1.922904117775608e-18im